Steam project

evaluated : 960 min 

get the data on : 

s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json

Synthesis Presentation : https://docs.google.com/presentation/d/1HcbFSU8F7MTcL2r48NYg0GMISnnjGq7CB0oxV1Mu5Vk/edit?slide=id.g1c7feb7f43d_0_37#slide=id.g1c7feb7f43d_0_37 

# Project : Steam's videogames platform 👾
Steam's videogames platform 👾
960 min

Steam
## Company's description 📇

Steam is a video game digital distribution service and storefront from Valve. It was launched as a software client in September 2003 to provide game updates automatically for Valve's games, and expanded to distributing third-party titles in late 2005. Steam offers various features, like digital rights management (DRM), game server matchmaking with Valve Anti-Cheat measures, social networking, and game streaming services. Steam client's functions include game update automation, cloud storage for game progress, and community features such as direct messaging, in-game overlay functions and a virtual collectable marketplace.
  
## Project 🚧

You're working for Ubisoft, a French video game publisher. They'd like to release a new revolutionary videogame! They asked you conduct a global analysis of the games available on Steam's marketplace in order to better understand the videogames ecosystem and today's trends.
  
  
# Goals 🎯

The ultimate goal of this project is to understand what factors affect the popularity or sales of a video game. But your boss asked you to take advantage of this opportunity to analyze the video game market globally.

To carry out this project, you will have to adopt different levels of analysis. Your boss gave you a list of examples of questions that would be interesting:

### Analysis at the "macro" level

    Which publisher has released the most games on Steam?

    What are the best rated games?

    Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

    How are the prizes distributed? Are there many games with a discount?

    What are the most represented languages?

    Are there many games prohibited for children under 16/18?

### Genres analysis

    What are the most represented genres?

    Are there any genres that have a better positive/negative review ratio?

    Do some publishers have favorite genres?

    What are the most lucrative genres?

### Platform analysis

    Are most games available on Windows/Mac/Linux instead?

    Do certain genres tend to be preferentially available on certain platforms?

You're free to follow these guidelines, or to choose a different angle of analysis, as long as your analysis reveals relevant and useful information. 🤓
  
  
## Scope of this project 🖼️

You'll have to use Databricks and PySpark to conduct this EDA. Particularly, you'll have to use Databrick's visualisation tool to create the visualizations.

The dataset is available in our S3 bucket at the following url: s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json.


In [0]:
pip list

Package                            Version
---------------------------------- -------------------------------
aiohappyeyeballs                   2.4.4
aiohttp                            3.11.10
aiosignal                          1.2.0
annotated-doc                      0.0.4
annotated-types                    0.7.0
anyio                              4.7.0
argon2-cffi                        21.3.0
argon2-cffi-bindings               21.2.0
arro3-core                         0.6.5
arrow                              1.3.0
asttokens                          3.0.0
astunparse                         1.6.3
async-lru                          2.0.4
attrs                              24.3.0
azure-common                       1.1.28
azure-core                         1.37.0
azure-identity                     1.20.0
azure-mgmt-core                    1.6.0
azure-mgmt-web                     8.0.0
azure-storage-blob                 12.28.0
azure-storage-file-datalake        12.22.0
babel            

In [0]:
# --------------------------------------------------------------------------
# # Part 1 : ETL : load dataset from json file in S3 bucket : 
# --------------------------------------------------------------------------

# Import PySpark functions
from pyspark.sql.functions import (
    col, explode, split, try_to_date, year, count,
    desc, avg, sum as _sum, when, lit, round, regexp_extract
)
from pyspark.sql import functions as F

# -- 1. Load data file from S3 bucket : --
file_path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df_raw = spark.read.json(file_path)




In [0]:
# Let's explore this data structure: 
print(f" Length : {df_raw.count()}\n Number of columns : {len(df_raw.columns)}")
print(f" Stats: \n")
df_raw.describe().show()

print("Schema :")
df_raw.printSchema()

 Length : 55691
 Number of columns : 2
 Stats: 

+-------+------------------+
|summary|                id|
+-------+------------------+
|  count|             55691|
|   mean|1025603.0926720655|
| stddev|522784.96832834254|
|    min|                10|
|    max|            999990|
+-------+------------------+

Schema :
root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |

In [0]:
df = df_raw.select("data.*")
df.printSchema()
df.show(5, truncate=False)

root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- tags: struct (nullable = true)
 |    |-- 1980s: lon

In [0]:
# We shall explore sub structures : 
# - categories
# - platforms
# - tags

# -- 2. Flatten the root struct data : --
df = df_raw.select("data.*")
df.printSchema()



root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- tags: struct (nullable = true)
 |    |-- 1980s: lon

In [0]:
df.show(5, truncate=False)
tag_cols = df.select("tags.*").columns
print(len(tag_cols), tag_cols[:20])

+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------------------+--------+-----------------------------------+-----------------------------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------------------------------+---------------------------+--------+------------------------+--------------------+--------+-----+------------------------+------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Interesting columns :  
categories (list of categories)  
owners : of type : 0 .. 20,000  ; 200,000 .. 500,000
ccu : concurent users ; int  
platforms : {3 booleens}  {true, true, true}
tags : of type : {266, 1191, NULL, NULL, NULL, ... }  

In [0]:
# --- 3. Global view ---
print(df.count(), len(df.columns))
df.describe().show()

55691 22
+-------+------------------+-----------------+------------------+------------------+--------------+--------------------+-----------------+---------+------------------------+------------------+--------------------+------------------+-----------------+--------------------+------------+------------------+--------------------+--------+------------------+
|summary|             appid|              ccu|         developer|          discount|         genre|        header_image|     initialprice|languages|                    name|          negative|              owners|          positive|            price|           publisher|release_date|      required_age|   short_description|    type|           website|
+-------+------------------+-----------------+------------------+------------------+--------------+--------------------+-----------------+---------+------------------------+------------------+--------------------+------------------+-----------------+--------------------+------------+-

In [0]:
# --- 4. Add columns from Struct simple "platforms" : ---
df = df.select("*", "platforms.*").drop("platforms")
df.groupBy("windows", "mac", "linux").count().show()

+-------+-----+-----+-----+
|windows|  mac|linux|count|
+-------+-----+-----+-----+
|   true| true|false| 5951|
|   true| true| true| 6807|
|   true|false| true| 1647|
|  false| true|false|   11|
|   true|false|false|41271|
|  false|false| true|    3|
|  false| true| true|    1|
+-------+-----+-----+-----+



In [0]:
# --- 5. Add dictionary from Struct "tags" many columns !! ---
tag_cols = df.select("tags.*").columns
print(len(tag_cols), tag_cols[:20])

441 ['1980s', "1990's", '2.5D', '2D', '2D Fighter', '2D Platformer', '360 Video', '3D', '3D Fighter', '3D Platformer', '3D Vision', '4 Player Local', '4X', '6DOF', '8-bit Music', 'ATV', 'Abstract', 'Action', 'Action RPG', 'Action RTS']


Tags is composed of 441 columns with a rating for each :  
multidimensionnal rating for each game !!!  

In [0]:
# count number of non null values :
df.select(F.count(F.col("tags").getField("1980s"))).show()

+-----------------+
|count(tags.1980s)|
+-----------------+
|             1101|
+-----------------+



In [0]:
df.select(F.count(F.lit(1)).alias("nb_total_lignes")).show()

+---------------+
|nb_total_lignes|
+---------------+
|          55691|
+---------------+



In [0]:
# Récupère la première ligne du struct tags sous forme de dict
premiere_ligne_tags = df.select("tags").first()["tags"].asDict()

# Compte les valeurs non-null
nb_non_null = sum(1 for v in premiere_ligne_tags.values() if v is not None)
nb_null = sum(1 for v in premiere_ligne_tags.values() if v is None)

print(f"Non-null : {nb_non_null}")
print(f"Null : {nb_null}")
print(f"Total : {len(premiere_ligne_tags)}")

Non-null : 20
Null : 421
Total : 441


So, we have the confirmation that we have 441 tags described for the element of first line of the dataset !! 

Tags columns will give a rating per genre , that's more precise,  
Let's get these dictionnaries  
We'll extract these tags for future usage !


In [0]:
df = df.drop("tags_dict")

In [0]:
# function to extract tags as dicts in a column "tags_dict" :

from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import MapType, StringType, LongType


def extraire_tags_dict(
    df: DataFrame,
    struct_col: str = "tags",
    name_col: str = "name",
    out_col: str = "tags_dict",
    value_type=LongType(),
    ) -> DataFrame:
    """
    Convertit une colonne struct (tags imbriqués avec valeurs numériques)
    en une colonne de type map<string, value_type>, en filtrant les entrées NULL.

    Parametres
    ----------
    df : DataFrame source
    struct_col : chemin de la colonne struct à convertir (ex: "data.tags")
    name_col : chemin de la colonne identifiant le jeu (ex: "data.name")
    out_col : nom de la colonne map en sortie
    value_type : type des valeurs du struct (LongType par défaut, adapter si besoin)

    Retour
    ------
    DataFrame avec [name_col, out_col, nb_tags]
    """
    if struct_col not in [f.name for f in df.schema.fields] and "." in struct_col:
        # struct_col est un chemin imbriqué (ex: "data.tags") -> on vérifie via select direct
        pass

    df_out = (
        df
        .withColumn("_tags_json", F.to_json(F.col(struct_col)))
        .withColumn(
            out_col,
            F.map_filter(
                F.from_json(F.col("_tags_json"), MapType(StringType(), value_type)),
                lambda k, v: v.isNotNull(),
            ),
        )
        .withColumn("nb_tags", F.size(F.col(out_col)))
        .drop("_tags_json").drop("tags")
    )

    return df_out


# --- Usage ---
df = extraire_tags_dict(df, struct_col="tags", name_col="name")
df_tags_dict = df.orderBy(F.desc("nb_tags"))
df_tags_dict.show(10)
df_tags_dict.groupBy("nb_tags").count().orderBy("nb_tags").show()
# --- 6. Add columns from Struct "genres" : ---
genre_cols = df.select("genres.*")

# --- Bonus : récupérer un échantillon en dict Python natif ---
def echantillon_dicts(df_tags_dict: DataFrame, n: int = 5, name_col: str = "name", out_col: str = "tags_dict"):
    """Retourne une liste de (nom_jeu, dict_python) pour inspection rapide."""
    rows = df_tags_dict.limit(n).collect()
    return [(row[name_col], dict(row[out_col])) for row in rows]

for nom, tags in echantillon_dicts(df_tags_dict):
    print(nom, "->", tags)

+-------+--------------------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+-----+-----+-------+--------------------+-------+
|  appid|          categories|ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|            owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|linux|  mac|windows|           tags_dict|nb_tags|
+-------+--------------------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+-----+

In [0]:
df.show(10)

+-------+--------------------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+--------------------+------------+------------+-------------------------------------+----+--------------------+-----+-----+-------+--------------------+-------+
|  appid|          categories|  ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                                name|negative|              owners|positive|price|           publisher|release_date|required_age|                    short_description|type|             website|linux|  mac|windows|           tags_dict|nb_tags|
+-------+--------------------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+------

In [0]:
# 6 -- extract owners : 

df = df.withColumn(
    "owners_min",
    F.regexp_extract(F.col("owners"), r"^([\d,]+)\s*\.\.\s*([\d,]+)$", 1)
).withColumn(
    "owners_max",
    F.regexp_extract(F.col("owners"), r"^([\d,]+)\s*\.\.\s*([\d,]+)$", 2)
)

# Nettoyage des virgules puis conversion en long
df = df.withColumn(
    "owners_min", F.regexp_replace(F.col("owners_min"), ",", "").cast("long")
).withColumn(
    "owners_max", F.regexp_replace(F.col("owners_max"), ",", "").cast("long")
).withColumn(
    "owners_mean",F.expr("floor((owners_min + owners_max) / 2)")
)

df.select("owners", "owners_min", "owners_max","owners_mean").show(10, truncate=False)

+------------------------+----------+----------+-----------+
|owners                  |owners_min|owners_max|owners_mean|
+------------------------+----------+----------+-----------+
|10,000,000 .. 20,000,000|10000000  |20000000  |15000000   |
|0 .. 20,000             |0         |20000     |10000      |
|200,000 .. 500,000      |200000    |500000    |350000     |
|100,000 .. 200,000      |100000    |200000    |150000     |
|0 .. 20,000             |0         |20000     |10000      |
|100,000 .. 200,000      |100000    |200000    |150000     |
|0 .. 20,000             |0         |20000     |10000      |
|20,000 .. 50,000        |20000     |50000     |35000      |
|0 .. 20,000             |0         |20000     |10000      |
|0 .. 20,000             |0         |20000     |10000      |
+------------------------+----------+----------+-----------+
only showing top 10 rows


In [0]:
df.drop("owners")


DataFrame[appid: bigint, categories: array<string>, ccu: bigint, developer: string, discount: string, genre: string, header_image: string, initialprice: string, languages: string, name: string, negative: bigint, positive: bigint, price: string, publisher: string, release_date: string, required_age: string, short_description: string, type: string, website: string, linux: boolean, mac: boolean, windows: boolean, tags_dict: map<string,bigint>, nb_tags: int, owners_min: bigint, owners_max: bigint, owners_mean: bigint]

In [0]:
# we are looking for any column with sale, sales, Sales *sales* ...  : 

from pyspark.sql.types import StructType, ArrayType

def trouver_colonnes(schema, motif, prefixe=""):
    """
    Parcourt récursivement un schéma Spark et renvoie
    les chemins complets des colonnes contenant `motif` (insensible à la casse).
    """
    resultats = []
    for champ in schema.fields:
        chemin = f"{prefixe}.{champ.name}" if prefixe else champ.name

        # Vérifie le nom du champ courant
        if motif.lower() in champ.name.lower():
            resultats.append(chemin)

        # Struct imbriqué -> on descend
        if isinstance(champ.dataType, StructType):
            resultats += trouver_colonnes(champ.dataType, motif, chemin)

        # Array de struct -> on descend aussi (via .element)
        elif isinstance(champ.dataType, ArrayType) and isinstance(champ.dataType.elementType, StructType):
            resultats += trouver_colonnes(champ.dataType.elementType, motif, chemin + "[]")

    return resultats
colonnes_sale = trouver_colonnes(df_raw.schema, "Sale")
print(f"colonnes sale:\n {colonnes_sale}")

colonnes_price = trouver_colonnes(df_raw.schema, "price")
print(f"Colonnes price:\n {colonnes_price}")



colonnes sale:
 []
Colonnes price:
 ['data.initialprice', 'data.price']


There is not any column with any staing *sale[s]* 

In [0]:
df.describe().toPandas()

,summary,appid,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,positive,price,publisher,release_date,required_age,short_description,type,website,nb_tags,owners_min,owners_max,owners_mean
0,count,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691
1,mean,1025603.0926720655,138.9596164550825,67391.3,2.603777989262179,None,None,797.5663033524268,None,Infinity,241.8376937027527,None,1470.8755992889335,773.2849832109317,2498.75,None,0.1978882344490734,None,None,None,12.129266847425976,78868.21928139197,188181.03463755365,133524.6269594728
2,stddev,522784.96832834254,6002.067909130784,210681.95381245477,12.887080174743156,None,None,1104.762477841305,None,NaN,5765.4137615598665,None,30982.733479535047,1093.134582723431,1809.1987867561706,None,2.296292461481821,None,None,None,6.733268747492117,1095229.2320051277,2607885.868024725,1849909.8295010144
3,min,10,0,,0,,https://cdn.akamai.steamstatic.com/steam/apps/...,0,,Fieldrunners 2,0,"0 .. 20,000",0,0,,,0,,game,,0,0,20000,10000
4,max,2190950,874053,＼上／,90,Web Publishing,https://cdn.akamai.steamstatic.com/steam/apps/...,9999,Turkish,～Daydream～蝶が舞う頃に,908515,"500,000 .. 1,000,000",5943345,9999,Ｌｅｍｏｎ Ｂａｌｍ,2022/11/7,MA 15+,🚗 Take part in a roller coaster of emotions wi...,hardware,www.windybeard.com,21,200000000,500000000,350000000


In [0]:
# Mapping names : => df_flat

mapping = {
    "ccu" :"concurrent_users",
    "genre" : "genre_string",
    "initialprice" :"price_raw",
    "release_date" :"date_raw",
    "required_age" : "required_age_raw",
    "languages" : "languages_string",
    "positive" : "positive_reviews",
    "negative" : "negative_reviews"
}

for old, new in mapping.items():
    df = df.withColumnRenamed(old,new)
    
df_flat = df.select("*")

In [0]:
df_cat = df_flat

df_cat = df_cat.withColumn("nb_categories", F.size("categories"))
df_cat = df_cat.select("appid","name",F.explode("categories").alias("category"))
display(df_cat)

df_cat_count = df_cat.groupBy("category").count().orderBy(F.desc("count"))
display(df_cat_count.limit(10))

appid,name,category
10,Counter-Strike,Multi-player
10,Counter-Strike,Valve Anti-Cheat enabled
10,Counter-Strike,Online PvP
10,Counter-Strike,Shared/Split Screen PvP
10,Counter-Strike,PvP
1000000,ASCENXION,Single-player
1000000,ASCENXION,Partial Controller Support
1000000,ASCENXION,Steam Achievements
1000000,ASCENXION,Steam Cloud
1000010,Crown Trick,Single-player


category,count
Single-player,52025
Steam Achievements,27394
Steam Cloud,14235
Full controller support,11879
Multi-player,11455
Steam Trading Cards,9208
Partial Controller Support,7867
PvP,7070
Co-op,5616
Steam Leaderboards,5509


Databricks visualization. Run in Databricks to view.

In [0]:

df_flat.show(10)

+-------+--------------------+----------------+--------------------+--------+--------------------+--------------------+---------+--------------------+------------------------------------+----------------+--------------------+----------------+-----+--------------------+----------+----------------+-------------------------------------+----+--------------------+-----+-----+-------+--------------------+-------+----------+----------+-----------+
|  appid|          categories|concurrent_users|           developer|discount|        genre_string|        header_image|price_raw|    languages_string|                                name|negative_reviews|              owners|positive_reviews|price|           publisher|  date_raw|required_age_raw|                    short_description|type|             website|linux|  mac|windows|           tags_dict|nb_tags|owners_min|owners_max|owners_mean|
+-------+--------------------+----------------+--------------------+--------+--------------------+------------

In [0]:
df.sample(False, 0.001).show()

+-------+--------------------+----------------+--------------------+--------+--------------------+--------------------+---------+--------------------+--------------------+----------------+--------------------+----------------+-----+--------------------+----------+----------------+--------------------+----+--------------------+-----+-----+-------+--------------------+-------+----------+----------+-----------+
|  appid|          categories|concurrent_users|           developer|discount|        genre_string|        header_image|price_raw|    languages_string|                name|negative_reviews|              owners|positive_reviews|price|           publisher|  date_raw|required_age_raw|   short_description|type|             website|linux|  mac|windows|           tags_dict|nb_tags|owners_min|owners_max|owners_mean|
+-------+--------------------+----------------+--------------------+--------+--------------------+--------------------+---------+--------------------+--------------------+-----

In [0]:
df_flat.describe().show()

+-------+------------------+-----------------+------------------+------------------+--------------+--------------------+-----------------+----------------+------------------------+------------------+--------------------+------------------+-----------------+--------------------+---------+------------------+--------------------+--------+------------------+------------------+------------------+------------------+------------------+
|summary|             appid| concurrent_users|         developer|          discount|  genre_string|        header_image|        price_raw|languages_string|                    name|  negative_reviews|              owners|  positive_reviews|            price|           publisher| date_raw|  required_age_raw|   short_description|    type|           website|           nb_tags|        owners_min|        owners_max|       owners_mean|
+-------+------------------+-----------------+------------------+------------------+--------------+--------------------+--------------

In [0]:
df_flat.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_flat.columns
]).show(truncate=False)

+-----+----------+----------------+---------+--------+------------+------------+---------+----------------+----+----------------+------+----------------+-----+---------+--------+----------------+-----------------+----+-------+-----+---+-------+---------+-------+----------+----------+-----------+
|appid|categories|concurrent_users|developer|discount|genre_string|header_image|price_raw|languages_string|name|negative_reviews|owners|positive_reviews|price|publisher|date_raw|required_age_raw|short_description|type|website|linux|mac|windows|tags_dict|nb_tags|owners_min|owners_max|owners_mean|
+-----+----------+----------------+---------+--------+------------+------------+---------+----------------+----+----------------+------+----------------+-----+---------+--------+----------------+-----------------+----+-------+-----+---+-------+---------+-------+----------+----------+-----------+
|0    |0         |0               |0        |0       |0           |0           |0        |0               |0 

## Transform and feature engineering

In [0]:
# -- 3. Transformations et feature engineering --
from pyspark.sql import functions as F
import math
z_val=1.96
z_sq = z_val ** 2

df_clean = df_flat.withColumn("discount", F.col("discount").cast("integer")
).withColumn("price_raw", F.col("price_raw").cast("float")
).withColumn("price", F.col("price").cast("float")
).withColumn(
    #"release_year", F.year(F.to_date(F.col("date_raw"), "yyyy/MM/d"))
    "release_year", 
    F.regexp_extract(F.col("date_raw"), r"(\d{4})", 1).try_cast("integer")
    #).withColumn(F.to_date(F.col("date_raw"), "yyyy-MM-dd"))
).withColumn(
    # Nettoyage age : extraction des chiffres (ex: "MA 15+" -> 15)
    "required_age",
    regexp_extract(col("required_age_raw"), r"(\d+)", 1).cast("integer")
).withColumn(
    # Valeur par defaut 0 si pas d'age requis
    "required_age",
    when(col("required_age").isNull(), 0).otherwise(col("required_age"))
).withColumn(
    "total_reviews", col("positive_reviews") + col("negative_reviews")
).withColumn(
    "review_rate",
    when(
        col("total_reviews") > 0,
        round((col("positive_reviews") / col("total_reviews")), 2)
        ).otherwise(0)
).withColumn("review_score_pond",
            (col("review_rate") - (col("review_rate") - 0.5) * F.pow(F.lit(2),-F.log10(col("total_reviews")+1)))*100
).withColumn(
    "wilson_score",
    when(col("total_reviews")>0,
             ((
                 col("review_rate")
                 + F.lit(z_sq) / (2 * col("total_reviews")) 
                 - F.lit(z_val) * F.sqrt(
                    (F.col("review_rate") * (1 - F.col("review_rate")) + F.lit(z_sq) / (4 * F.col("total_reviews"))) 
                    / col("total_reviews")
                 )
            ) / (1 + F.lit(z_sq) / F.col("total_reviews"))
             )*100  
    ).otherwise(0.0)         
             

).withColumn(
    "review_rate", col("review_rate") *100 

).withColumn("revenue", col("price") * col("owners_mean")
)
#wilson_score = (phat + z**2 / (2 * total) - z * math.sqrt((phat * (1 - phat) + z**2 / (4 * total)) / total)     ) / (1 + z**2 / total)
    


display(df_clean.limit(10))



appid,categories,concurrent_users,developer,discount,genre_string,header_image,price_raw,languages_string,name,negative_reviews,owners,positive_reviews,price,publisher,date_raw,required_age_raw,short_description,type,website,linux,mac,windows,tags_dict,nb_tags,owners_min,owners_max,owners_mean,release_year,required_age,total_reviews,review_rate,review_score_pond,wilson_score,revenue
10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999.0,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000",201215,999.0,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,game,,true,true,true,"Map(1980s -> 266, PvP -> 881, Team-Based -> 1864, 1990's -> 1191, Old School -> 769, Competitive -> 1607, Score Attack -> 289, FPS -> 4831, Tactical -> 1344, Strategy -> 614, Nostalgia -> 131, Assassin -> 227, Multiplayer -> 3392, Shooter -> 3353, First-Person -> 1707, Survival -> 304, Military -> 632, Classic -> 2784, e-sports -> 1192, Action -> 5426)",20,10000000,20000000,15000000,2000,0,206414,97.0,95.81913208871065,96.92552837514519,1.4985E10
1000000,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",0,IndigoBlue Game Studio,0,"Action, Adventure, Indie",https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048,999.0,"English, Korean, Simplified Chinese",ASCENXION,5,"0 .. 20,000",27,999.0,PsychoFlux Entertainment,2021/05/14,0,"ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, to uncover the truth of this world.",game,,false,false,true,"Map(2D -> 159, Indie -> 69, Robots -> 100, Bullet Hell -> 179, Twin Stick Shooter -> 170, Side Scroller -> 175, Minimalist -> 136, Great Soundtrack -> 38, GameMaker -> 51, Controller -> 148, Colorful -> 124, Singleplayer -> 71, Abstract -> 111, Shooter -> 159, Difficult -> 161, Shoot 'Em Up -> 186, Metroidvania -> 181, Adventure -> 73, Atmospheric -> 88, Action -> 138)",20,0,20000,10000,2021,0,32,84.0,72.13246087153605,67.81250851505555,9990000.0
1000010,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)",99,NEXT Studios,70,"Adventure, Indie, RPG, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/1000010/header.jpg?t=1655724189,1999.0,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",Crown Trick,646,"200,000 .. 500,000",4032,599.0,"Team17, NEXT Studios",2020/10/16,0,"Enter a labyrinth that moves as you move, where mastering the elements is key to defeating enemies and uncovering the mysteries of this underground world. With a new experience awaiting every time you enter the dungeon, let the power bestowed by the crown guide you in this challenging adventure!",game,,false,false,true,"Map(2D -> 205, Rogue-lite -> 226, Rogue-like -> 268, Magic -> 171, Replay Value -> 192, Fantasy -> 179, Dungeon Crawler -> 225, Perma Death -> 231, Tactical -> 184, Turn-Based -> 222, Strategy -> 225, Top-Down -> 178, RPG -> 237, Singleplayer -> 211, Difficult -> 189, Turn-Based Combat -> 254, Female Protagonist -> 217, Adventure -> 225, Action-Adventure -> 179, Turn-Based Strategy -> 216)",20,200000,500000,350000,2020,0,4678,86.0,83.17201992340665,84.97608020807645,2.0965E8
1000030,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, F

In [0]:
df_clean.show(15)

+-------+--------------------+----------------+----------------------------+--------+--------------------+--------------------+---------+--------------------+------------------------------------+----------------+--------------------+----------------+------+----------------------------+----------+----------------+-------------------------------------+----+--------------------+-----+-----+-------+--------------------+-------+----------+----------+-----------+------------+------------+-------------+-----------+------------------+------------------+---------+
|  appid|          categories|concurrent_users|                   developer|discount|        genre_string|        header_image|price_raw|    languages_string|                                name|negative_reviews|              owners|positive_reviews| price|                   publisher|  date_raw|required_age_raw|                    short_description|type|             website|linux|  mac|windows|           tags_dict|nb_tags|owners_min|

In [0]:
# -- 4. Create specialised views for ML --

# View by genre (several genres are possible for one game !)
df_genres_exploded = df_clean.select(
    "name", "publisher", "review_rate", "total_reviews", "linux","mac","windows",
    "positive_reviews", "negative_reviews","owners_mean","review_score_pond","wilson_score","revenue",
    explode(split(col("genre_string"), ", ")).alias("genre")
)
print(f" View of games by genre  first 8 lines: ")
df_genres_exploded.show(8)

# View by language
df_languages = df_clean.select(
    "name",
    explode(split(col("languages_string"), ", ")).alias("language")
)
print(f" View of games by language top 7 lines : ")
df_languages.show(7)

print(f"Pipeline ETL upon to end. {df_clean.count()} games ready.")
print(f"first 8 rows of df_clean dataframe: ")
display(df_clean.select("name", "required_age", "required_age_raw").limit(5))

# selection d'un subset ou required_age est non 0
sub = df_clean.filter(col("required_age") != 0)
print (f" View of games where required age non zero:" )
display(sub.select("name", "required_age", "required_age_raw").limit(5))


# View by categories : 
df_clean = df_clean.withColumn("nb_categories", F.size("categories"))
cat_columns = ["appid","name", "publisher", "review_rate", "total_reviews", "linux","mac","windows",
    "positive_reviews", "negative_reviews","owners_mean","review_score_pond","wilson_score","revenue"]
    #"categories"


df_cat = df_clean.select("appid","name", "publisher", "review_rate", "total_reviews", "linux","mac","windows",
    "positive_reviews", "negative_reviews","owners_mean","review_score_pond","wilson_score","revenue",F.explode("categories").alias("category"))
display(df_cat)

df_cat_count = df_cat.groupBy("category").count().orderBy(F.desc("count"))
display(df_cat_count.limit(10))

 View of games by genre  first 8 lines: 
+--------------+--------------------+-----------+-------------+-----+-----+-------+----------------+----------------+-----------+-----------------+-----------------+---------+---------+
|          name|           publisher|review_rate|total_reviews|linux|  mac|windows|positive_reviews|negative_reviews|owners_mean|review_score_pond|     wilson_score|  revenue|    genre|
+--------------+--------------------+-----------+-------------+-----+-----+-------+----------------+----------------+-----------+-----------------+-----------------+---------+---------+
|Counter-Strike|               Valve|       97.0|       206414| true| true|   true|          201215|            5199|   15000000|95.81913208871065|96.92552837514519|1.4985E10|   Action|
|     ASCENXION|PsychoFlux Entert...|       84.0|           32|false|false|   true|              27|               5|      10000|72.13246087153605|67.81250851505555|9990000.0|   Action|
|     ASCENXION|PsychoFlux En

name,required_age,required_age_raw
Counter-Strike,0,0
ASCENXION,0,0
Crown Trick,0,0
"Cook, Serve, Delicious! 3?!",0,0
细胞战争,0,0


 View of games where required age non zero:


name,required_age,required_age_raw
Blacksad: Under the Skin,15,15
EARTH DEFENSE FORCE 5,15,15
Settlement Zero,12,12
Funbag Fantasy,18,18
Cyndy,18,18


appid,name,publisher,review_rate,total_reviews,linux,mac,windows,positive_reviews,negative_reviews,owners_mean,review_score_pond,wilson_score,revenue,category
10,Counter-Strike,Valve,97.0,206414,true,true,true,201215,5199,15000000,95.81913208871065,96.92552837514519,1.4985E10,Multi-player
10,Counter-Strike,Valve,97.0,206414,true,true,true,201215,5199,15000000,95.81913208871065,96.92552837514519,1.4985E10,Valve Anti-Cheat enabled
10,Counter-Strike,Valve,97.0,206414,true,true,true,201215,5199,15000000,95.81913208871065,96.92552837514519,1.4985E10,Online PvP
10,Counter-Strike,Valve,97.0,206414,true,true,true,201215,5199,15000000,95.81913208871065,96.92552837514519,1.4985E10,Shared/Split Screen PvP
10,Counter-Strike,Valve,97.0,206414,true,true,true,201215,5199,15000000,95.81913208871065,96.92552837514519,1.4985E10,PvP
1000000,ASCENXION,PsychoFlux Entertainment,84.0,32,false,false,true,27,5,10000,72.13246087153605,67.81250851505555,9990000.0,Single-player
1000000,ASCENXION,PsychoFlux Entertainment,84.0,32,false,false,true,27,5,10000,72.13246087153605,67.81250851505555,9990000.0,Partial Controller Support
1000000,ASCENXION,PsychoFlux Entertainment,84.0,32,false,false,true,27,5,10000,72.13246087153605,67.81250851505555,9990000.0,Steam Achievements
1000000,ASCENXION,PsychoFlux Entertainment,84.0,32,false,false,true,27,5,10000,72.13246087153605,67.81250851505555,9990000.0,Steam Cloud
1000010,Crown Trick,"Team17, NEXT Studios",86.0,4678,false,false,true,4032,646,350000,83.17201992340665,84.97608020807645,2.0965E8,Single-player


category,count
Single-player,52025
Steam Achievements,27394
Steam Cloud,14235
Full controller support,11879
Multi-player,11455
Steam Trading Cards,9208
Partial Controller Support,7867
PvP,7070
Co-op,5616
Steam Leaderboards,5509


### 1 Analysis at the "macro" level
#### 1.1 Which publisher has released the most games on Steam?  


In [0]:
#### Which publisher has released the most games on Steam?
publisher_max = df_clean.select('publisher','name',"date_raw").groupBy(   
    'publisher').agg(
        F.count('date_raw').alias('count_releases'),
        F.count_distinct('name').alias('count_games')
        ).orderBy(F.desc('count_releases'))
print(f"Top 7 Publishers who have released most games on Steam:")
display(publisher_max.limit(10))
"""fig = px.bar(publisher_max.limit(12), x='publisher', y='count_releases',
             
        title="Publishers games releases on Steam",
        color='publisher'
       )
fig.show()"""


Top 7 Publishers who have released most games on Steam:


publisher,count_releases,count_games
Big Fish Games,422,422
8floor,202,202
SEGA,165,165
Strategy First,151,151
Square Enix,141,141
Choice of Games,140,140
HH-Games,132,132
,132,132
Sekai Project,132,132
Ubisoft,127,127


Databricks visualization. Run in Databricks to view.

'fig = px.bar(publisher_max.limit(12), x=\'publisher\', y=\'count_releases\',\n             \n        title="Publishers games releases on Steam",\n        color=\'publisher\'\n       )\nfig.show()'

In [0]:
#display(df_clean.filter(col("publisher")=="Big Fish Games").limit(25))


In [0]:
# What are the best rated games?
best_rated = df_clean.select('publisher','name','positive_reviews').orderBy(F.desc('positive_reviews')).limit(20)
#.groupBy('name','publisher').agg(F.sum('positive_reviews').alias('sum_positive_reviews'))
print(f"Top 5 Best rated games : ")
display(best_rated.limit(10))
"""px.bar(best_rated, x='title', y='sum_positive_reviews',
       color='publisher',
       title="Steam Best rated games",
       #category_orders = {'title': best_rated['title'].tolist()}
       )"""


Top 5 Best rated games : 


publisher,name,positive_reviews
Valve,Counter-Strike: Global Offensive,5943345
Valve,Dota 2,1534895
Rockstar Games,Grand Theft Auto V,1229265
"KRAFTON, Inc.",PUBG: BATTLEGROUNDS,1185361
Re-Logic,Terraria,1014711
Ubisoft,Tom Clancy's Rainbow Six Siege,942910
Valve,Garry's Mod,861240
Valve,Team Fortress 2,846407
Facepunch Studios,Rust,732513
Valve,Left 4 Dead 2,643836


Databricks visualization. Run in Databricks to view.

'px.bar(best_rated, x=\'title\', y=\'sum_positive_reviews\',\n       color=\'publisher\',\n       title="Steam Best rated games",\n       #category_orders = {\'title\': best_rated[\'title\'].tolist()}\n       )'

In [0]:
# what are the best scored games 
# rating and wilson : 
best_rated2 = df_clean.select('publisher','name',"review_score_pond","wilson_score","revenue").orderBy(F.desc('review_score_pond')).limit(20)
print(f"Top 5 Best ponderated-scored games")
display(best_rated2.limit(10))

Top 5 Best ponderated-scored games


publisher,name,review_score_pond,wilson_score,revenue
Valve,Portal 2,97.91014924500733,98.96432882377837,1.4985E10
Supergiant Games,Hades,97.7622985932789,98.95575589430766,1.87425E10
Studio Minus,People Playground,97.62956712721885,98.94739172669162,3.4965E9
poncle,Vampire Survivors,97.59131745242563,98.94486508776598,1.7465E9
Re-Logic,Terraria,97.25817786766567,97.97287681794228,3.4965E10
Wallpaper Engine Team,Wallpaper Engine,97.1127151145794,97.96339884718233,1.3965E10
ConcernedApe,Stardew Valley,97.07975500405121,97.96109137433807,2.2485E10
New Blood Interactive,ULTRAKILL,96.99139412675339,98.89846943250701,3.7485E9
Innersloth,The Henry Stickmin Collection,96.86870363027852,98.88740685294417,2.2485E9
"Nicalis, Inc.",The Binding of Isaac: Rebirth,96.80957150312634,97.94003100469641,1.12425E10


Databricks visualization. Run in Databricks to view.

Most expensive games are pretty confidential, or specific market ones ... 
Let's focus on  

In [0]:
# Are there years with more releases?
nb_release_year1 = df_clean.select("name","release_year").groupBy("release_year").agg(F.count("name").alias("count_release_year"))
nb_release_year = nb_release_year1.orderBy(F.desc("count_release_year"))
print(f"Top 5 years with the biggest number of releases :")
display(nb_release_year.limit(15))
#fig = px.line(nb_release_year1, x="release_year", y="count_release_year", title="Number of games releases per year")
# add 2 vertical red dashed lines between 2020 and 2022
#fig.add_vline(x=2020, line_dash="dash", line_color="red")
#fig.add_vline(x=2022, line_dash="dash", line_color="red")
#fig.show() 

Top 5 years with the biggest number of releases :


release_year,count_release_year
2021,8823
2020,8305
2018,7678
2022,7455
2019,6968
2017,6017
2016,4185
2015,2576
2014,1557
2013,471


Databricks visualization. Run in Databricks to view.

In [0]:
#  Were there more or fewer game releases during the Covid, for example? (between 2020 and 2022)
release_covid = df_clean.select("name","release_year").filter((col("release_year") >= 2018) & (col("release_year") <= 2022)).groupBy("release_year").agg(F.count("name").alias("count_release_year")).orderBy(F.desc("count_release_year")).limit(5)
print("Detailed view of Years during COVID with the numbers of releases")
display(release_covid)

# One can see below that number of releases has increased during covid located mainly in [2020 - 2022]

Detailed view of Years during COVID with the numbers of releases


release_year,count_release_year
2021,8823
2020,8305
2018,7678
2022,7455
2019,6968


Databricks visualization. Run in Databricks to view.

In [0]:
# What are the most represented languages?
languages_rep = df_languages.select("name","language").groupBy("language").agg(F.count("name").alias("number_games")).orderBy(F.desc("number_games")).limit(12)
print(f"Most represented languages : ")
display(languages_rep)
#fig.show()

Most represented languages : 


language,number_games
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

# How are the prices distributed? 

In [0]:
import plotly.express as px
# How are the prices distributed? 
# ===============================
# faire une loi de repartition type histogramme : 
prices_table = df_clean.select("name","price","price_raw", "publisher","release_year").orderBy(F.asc("price"), F.asc("price_raw"), F.asc("release_year"))

# --- 2. Bucket of price (50 buckets up to price_max) ---
# largeur de tranche = 10000 / 100 = 100 par tranche
nb_bins = 250
price_max = 10000
prices_table2 = prices_table.withColumn(
    "price_bucket_id",
    F.floor(F.least(F.col("price"), F.lit(price_max)) / nb_bins).cast("int")
).withColumn(
    "price_bucket",
    F.concat(
        (F.col("price_bucket_id") * nb_bins).cast("string"),
        F.lit(" - "),
        ((F.col("price_bucket_id") + 1) * nb_bins).cast("string")
    )
)

# --- 3. Bucket de price_raw (initial price) (nb_bins tranches jusqu'à 10000) ---
# largeur de tranche = 10000 / 100 = 100 par tranche
nb_bins = 200
price_max = 15000
prices_table2 = prices_table2.withColumn(
    "price_raw_bucket_id",
    F.floor(F.least(F.col("price_raw"), F.lit(price_max)) / nb_bins).cast("int")
).withColumn(
    "price_raw_bucket",
    F.concat(
        (F.col("price_raw_bucket_id") * nb_bins).cast("string"),
        F.lit(" - "),
        ((F.col("price_raw_bucket_id") + 1) * nb_bins).cast("string")
    )
)
# --- 3. Groupby price bucket 
print("Price repartition : ")
df_price_bucketed = (
    prices_table2.groupBy("price_bucket_id", "price_bucket", "price_raw_bucket_id","price_raw_bucket")
    .agg(F.count("name").alias("count_games"))
    .orderBy(F.asc("price_bucket_id"), F.asc("price_raw_bucket_id"))
)
# ,"release_year", "publisher"   , F.desc('release_year')

print(f"prices repartition for games which price < 10000" )
display(df_price_bucketed)
# histogram : X=price_eur, Y=count
# limit on X scale when no more data : 5000 , 
#fig = px.histogram(prices_table,x="price_eur",y="count",nbins = 30)
#print(f"Prices repartition : ")
#fig.show()



Price repartition : 
prices repartition for games which price < 10000


price_bucket_id,price_bucket,price_raw_bucket_id,price_raw_bucket,count_games
0,0 - 250,0,0 - 200,17594
0,0 - 250,1,200 - 400,298
0,0 - 250,2,400 - 600,338
0,0 - 250,3,600 - 800,84
0,0 - 250,4,800 - 1000,127
0,0 - 250,5,1000 - 1200,33
0,0 - 250,6,1200 - 1400,5
0,0 - 250,7,1400 - 1600,13
0,0 - 250,8,1600 - 1800,1
0,0 - 250,9,1800 - 2000,16


Databricks visualization. Run in Databricks to view.

In [0]:
prices_table.createOrReplaceTempView("prices_table_sql")

sql_result = spark.sql("""
    SELECT 
        histogram_numeric(price, 20) AS hist
    FROM prices_table_sql
    """)
display(sql_result)

hist
"List(List(240.89171, 33234.0), List(1073.1981, 16038.0), List(1962.7686, 3486.0), List(2747.0322, 1773.0), List(3863.6128, 594.0), List(4939.651, 255.0), List(5997.8994, 189.0), List(6893.0, 34.0), List(7905.25, 16.0), List(8915.667, 6.0), List(9926.621, 29.0), List(11999.0, 2.0), List(12915.667, 6.0), List(14984.857, 7.0), List(15999.0, 1.0), List(19999.0, 17.0), List(24900.0, 1.0), List(26999.0, 1.0), List(29990.0, 1.0), List(99900.0, 1.0))"


Databricks visualization. Run in Databricks to view.

In [0]:
prices_table.describe().show()

prices_table.select("price").filter(col("price") < 10000).plot.hist(bins=400)



+-------+------------------------+------------------+------------------+--------------------+------------------+
|summary|                    name|             price|         price_raw|           publisher|      release_year|
+-------+------------------------+------------------+------------------+--------------------+------------------+
|  count|                   55691|             55691|             55691|               55691|             55592|
|   mean|                Infinity| 773.2849832109317| 797.5663033524268|             2498.75|2018.6526298748022|
| stddev|                     NaN|1093.1345827234538|1104.7624778413326|  1809.1987867561706|  2.71834111468598|
|    min|          Fieldrunners 2|               0.0|               0.0|                    |              1997|
|    max|～Daydream～蝶が舞う頃に|           99900.0|           99900.0|Ｌｅｍｏｎ　Ｂａｌｍ|              2022|
+-------+------------------------+------------------+------------------+--------------------+------------------+



# answer how are Prices distributed :
Prices are mainly distributed along 2 families of meshes of 100 and 500  
slowly decreasing  
with a bump around 1000 and 1250  
    

# How is revenue distributed:  
revenue = price * nb_users_mean


In [0]:
revenue_table = df_clean.select("name","revenue","price","owners_mean","release_year").orderBy(F.desc("revenue")).limit(10)
print("Revenue repartition : ")
display(revenue_table)
revenue_table.describe().show()

df_clean.select("revenue").filter(col("revenue") < 10000).plot.hist(bins=400)


Revenue repartition : 


name,revenue,price,owners_mean,release_year
ELDEN RING,2.09965E11,5999.0,35000000,2022
New World,1.49925E11,1999.0,75000000,2021
Rust,1.39965E11,3999.0,35000000,2018
Grand Theft Auto V,1.0493E11,2998.0,35000000,2015
Cyberpunk 2077,8.9985E10,5999.0,15000000,2020
HITMAN 2,7.49925E10,9999.0,7500000,2018
Tom Clancy's Rainbow Six Siege,6.9965E10,1999.0,35000000,2015
Valheim,6.9965E10,1999.0,35000000,2021
The Witcher 3: Wild Hunt,5.9985E10,3999.0,15000000,2015
Mount & Blade II: Bannerlord,5.9985E10,3999.0,15000000,2022


Databricks visualization. Run in Databricks to view.

+-------+--------------+--------------------+------------------+--------------------+------------------+
|summary|          name|             revenue|             price|         owners_mean|      release_year|
+-------+--------------+--------------------+------------------+--------------------+------------------+
|  count|            10|                  10|                10|                  10|                10|
|   mean|          NULL|        1.0296625E11|            4298.9|             3.025E7|            2018.7|
| stddev|          NULL|4.927316687567721E10|2496.7223158017036|1.9237189099357642E7|2.9078437983419185|
|    min|Cyberpunk 2077|           5.9985E10|            1999.0|             7500000|              2015|
|    max|       Valheim|          2.09965E11|            9999.0|            75000000|              2022|
+-------+--------------+--------------------+------------------+--------------------+------------------+



In [0]:
prices_table.select("price").plot.box()

In [0]:
%sql
SELECT price FROM prices_table

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7085324146590276>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT price FROM prices_table\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_logger.logExec

In [0]:
display(prices_table)

"""fig = px.box(prices_table,y="price_raw")
print(f"Prices repartition : ")
fig.show()"""


In [0]:
# what are the most expensive games?
most_expensive = df_clean.select('publisher','name','price', "concurrent_users", "review_rate","owners_max", "positive_reviews").orderBy(F.desc('price')).limit(20)
print(f"Top 10 Most expensive games : ")
display(most_expensive.limit(10))


Top 10 Most expensive games : 


publisher,name,price,concurrent_users,review_rate,owners_max,positive_reviews
Fury Games,Ascent Free-Roaming VR Experience,99900.0,0,100.0,20000,6
Aartform,Aartform Curvy 3D 3.0,29990.0,0,70.0,20000,31
SideFX,Houdini Indie,26999.0,91,95.0,20000,152
MAGIX Software GmbH,VEGAS Edit 20 Steam Edition,24900.0,2,100.0,20000,1
"Kelley Integrity Safety Solutions, LLC, Endless Simulations",Spot Sample Witness Simulator,19999.0,0,71.0,20000,5
RK,VR Long March,19999.0,0,67.0,20000,2
wandwad,Lgnorant girl doll,19999.0,0,50.0,20000,2
Virtual Human Technologies,Virtual Orator,19999.0,0,100.0,20000,3
上海皋城软件有限公司,眼睛（眼球）结构研究,19999.0,0,100.0,20000,1
H.G.G.,Run Thief,19999.0,0,0.0,20000,0


In [0]:
display(df_clean.limit(10))

In [0]:
import numpy as np
# Are there many games with a discount?
discount = df_clean.select("name","price","discount").filter(col("discount") != 0).groupBy("price","discount").agg(F.count("discount").alias("count_discount")).orderBy(F.asc("price"),F.asc("discount"))

discount1 = df_clean.select("name","price","discount").filter(col("discount") != 0).orderBy(F.asc("price"),F.asc("discount"))

nb_discount = discount1.select(F.col('discount')).count()
nb_games = df_clean.select(F.col('name')).count()    
print(f"Number of games with a discount : {nb_discount}\n"
      f"Percentage of games with a discount : {np.round(nb_discount/nb_games*100,2)} %")
display(discount1)



print(f"Discount repartition : ")
discount2 = df_clean.select("name","price","discount").groupBy("price","discount").agg(F.count("discount").alias("count_discount")).orderBy(F.asc("price"),F.asc("discount"))
#display(discount2)

#fig = px.bar(discount, x="discount", y="count_discount",)

In [0]:
from pyspark.sql import functions as F

# --- 1. Bucket de discount (4 niveaux) ---
df_bucketed = df_clean.filter(col("discount") != 0).withColumn(
    "discount_bucket",
    F.when(F.col("discount") == 0, "0%")
     .when(F.col("discount") < 25, "1-24%")
     .when(F.col("discount") < 50, "25-49%")
     .when(F.col("discount") < 75, "50-74%")
     .otherwise("75%+")
)

# --- 2. Bucket de price (50 tranches jusqu'à 10000) ---
# largeur de tranche = 10000 / 100 = 100 par tranche
nb_bins = 50
price_max = 10000
df_bucketed = df_bucketed.withColumn(
    "price_bucket_id",
    F.floor(F.least(F.col("price"), F.lit(price_max)) / nb_bins).cast("int")
).withColumn(
    "price_bucket",
    F.concat(
        (F.col("price_bucket_id") * nb_bins).cast("string"),
        F.lit(" - "),
        ((F.col("price_bucket_id") + 1) * nb_bins).cast("string")
    )
)

# --- 3. Bucket de price_raw (initial price) (nb_bins tranches jusqu'à 10000) ---
# largeur de tranche = 10000 / 100 = 100 par tranche
nb_bins = 50
price_max = 10000
df_bucketed = df_bucketed.withColumn(
    "price_raw_bucket_id",
    F.floor(F.least(F.col("price_raw"), F.lit(price_max)) / nb_bins).cast("int")
).withColumn(
    "price_raw_bucket",
    F.concat(
        (F.col("price_raw_bucket_id") * nb_bins).cast("string"),
        F.lit(" - "),
        ((F.col("price_raw_bucket_id") + 1) * nb_bins).cast("string")
    )
)

# --- 3. Groupby sur les buckets ---
print("Discount repartition : ")
discount2 = (
    df_bucketed
    .groupBy("price_bucket_id", "price_bucket", "discount_bucket","price_raw_bucket_id","price_raw_bucket")
    .agg(F.count("discount").alias("count_discount"))
    .orderBy(F.asc("price_bucket_id"), F.asc("discount_bucket"),F.asc("price_raw_bucket_id"))
)

display(discount2)

In [0]:
# What are the most represented languages?
languages_rep = df_clean.select("name","languages")


In [0]:
# Are there many games prohibited for children under 16/18?

games_prohib = df_clean.select("name","required_age").filter(col("required_age") != 0).groupBy("required_age").agg(F.count("required_age").alias("count_required_age")).orderBy(F.desc("count_required_age")).limit(5)
print(f"games prohibed under 16/18 : {games_prohib.count()}")
display(games_prohib)  

games prohibed under 16/18 : 5


required_age,count_required_age
15,265
18,223
16,38
17,38
12,32


Databricks visualization. Run in Databricks to view.

### 2. Genres analysis  
    
#### 2.1 : What are the most represented genres?  

    - Are there any genres that have a better positive/negative review ratio?

    - Do some publishers have favorite genres?  

    - What are the most lucrative genres?  

In [0]:
df_genres_exploded.show(5)

+--------------+--------------------+-----------+-------------+-----+-----+-------+----------------+----------------+-----------+-----------------+-----------------+---------+---------+
|          name|           publisher|review_rate|total_reviews|linux|  mac|windows|positive_reviews|negative_reviews|owners_mean|review_score_pond|     wilson_score|  revenue|    genre|
+--------------+--------------------+-----------+-------------+-----+-----+-------+----------------+----------------+-----------+-----------------+-----------------+---------+---------+
|Counter-Strike|               Valve|       97.0|       206414| true| true|   true|          201215|            5199|   15000000|95.81913208871065|96.92552837514519|1.4985E10|   Action|
|     ASCENXION|PsychoFlux Entert...|       84.0|           32|false|false|   true|              27|               5|      10000|72.13246087153605|67.81250851505555|9990000.0|   Action|
|     ASCENXION|PsychoFlux Entert...|       84.0|           32|false|f

In [0]:
genres_rep.show(5)

In [0]:
# What are the most represented genres?  
genres_rep = df_genres_exploded.select("name","genre","total_reviews","review_rate","positive_reviews","negative_reviews").groupBy("genre").agg(
            F.count("genre").alias("nb_games"),
            F.sum("total_reviews").alias("sum_reviews"),
            F.avg("total_reviews").alias("avg_reviews"),
            F.avg("review_rate").alias("avg_review_rate"),
            F.sum("positive_reviews").alias("positive_reviews"),
            F.sum("negative_reviews").alias("negative_reviews"),
            ).orderBy(F.desc("nb_games"))
print(f"Most represented genres by number of games: ")
display(genres_rep)

# what are genres with best positive/negative reviews ratio:
print("Are there any genres that have a better positive/negative review ratio?")
genres_rep = genres_rep.withColumn(
    "pos_neg_reviews_ratio",
    col("positive_reviews") / col("negative_reviews")).orderBy(F.desc("pos_neg_reviews_ratio")).limit(10)
print(f"Genres with the best positive/negative review ratio: ")
display(genres_rep)    


Most represented genres by number of games: 


genre,nb_games,sum_reviews,avg_reviews,avg_review_rate,positive_reviews,negative_reviews
Indie,39681,36772257,926.6968322370908,73.990146417681,32531023,4241234
Action,23759,64546277,2716.708489414538,72.81585925333557,54858618,9687659
Casual,22086,11572263,523.9637326813366,74.02526487367564,10034967,1537296
Adventure,21431,35342598,1649.1343381083477,73.77107927768186,29689445,5653153
Strategy,10895,15796295,1449.8664525011473,71.818540614961,13402870,2393425
Simulation,10836,17972902,1658.6288298265042,69.20588778146917,15572390,2400512
RPG,9534,22699856,2380.9372771134886,72.96224040276904,19425528,3274328
Early Access,6145,5270786,857.7357200976404,70.47323026851099,4334595,936191
Free to Play,3393,23003053,6779.561744768641,72.09666961391099,18722246,4280807
Sports,2666,3672651,1377.5885221305327,69.6807951987997,2949805,722846


Databricks visualization. Run in Databricks to view.

Are there any genres that have a better positive/negative review ratio?
Genres with the best positive/negative review ratio: 


genre,nb_games,sum_reviews,avg_reviews,avg_review_rate,positive_reviews,negative_reviews,pos_neg_reviews_ratio
Photo Editing,105,591496,5633.295238095238,69.63809523809523,577751,13745,42.03353946889778
Animation & Modeling,322,717157,2227.195652173913,70.45962732919254,690765,26392,26.17327220369809
Design & Illustration,406,701064,1726.7586206896551,71.88916256157636,674057,27007,24.958603325063873
Utilities,682,782838,1147.8563049853371,69.76246334310851,739335,43503,16.99503482518447
Game Development,159,30735,193.30188679245282,73.77358490566037,27461,3274,8.38759926695174
Indie,39681,36772257,926.6968322370908,73.990146417681,32531023,4241234,7.67017877344188
Audio Production,195,78546,402.8,67.17948717948718,69118,9428,7.331141281289775
,161,217393,1350.2670807453417,67.84472049689441,189649,27744,6.835676182237601
Video Production,247,127876,517.7165991902834,67.74089068825911,111514,16362,6.815425987043149
Casual,22086,11572263,523.9637326813366,74.02526487367564,10034967,1537296,6.527673915758578


Databricks visualization. Run in Databricks to view.

In [0]:

# Do some publishers have favorite genres?  
genres_rep3 = df_genres_exploded.filter(col("publisher").isin(["Ubisoft","Big Fish Games","8floor","SEGA","Strategy First","Square Enix",
                "Choice of Games","Sekai Project"])).select(
                "name","genre","publisher").groupBy("genre","publisher").agg(
            F.count("genre").alias("nb_games"),
        ).orderBy(F.desc("nb_games"))

display(genres_rep3)            



  

genre,publisher,nb_games
Casual,Big Fish Games,418
Adventure,Big Fish Games,392
Casual,8floor,202
RPG,Choice of Games,139
Indie,Choice of Games,136
Adventure,Choice of Games,112
Casual,Sekai Project,99
Indie,Sekai Project,88
Action,SEGA,80
Action,Square Enix,75


Databricks visualization. Run in Databricks to view.

From former graph, one can see that 8Floor have a prefered genre : casual; 
Big fish Games prefer casual and adventure

#### 2.2 Are there any genres that have a better positive/negative review ratio?

In [0]:
# What are the most lucrative genres?  
genres_rep4 = df_genres_exploded.select("name","genre","revenue").groupBy("genre").agg(
            count("name").alias("nb_games"),
            avg("revenue").alias("avg_revenue"),
            ).orderBy(F.desc("avg_revenue")).limit(10)
print(f"Most lucrative genres by average revenue: ")
display(genres_rep4)


Most lucrative genres by average revenue: 


genre,nb_games,avg_revenue
Massively Multiplayer,1460,4.061751883561644E8
RPG,9534,2.850130385986994E8
Action,23759,2.4730188181320763E8
Strategy,10895,1.8494760027535567E8
Adventure,21431,1.7379374947505948E8
Simulation,10836,1.7321658868586195E8
Photo Editing,105,1.6644366666666666E8
,161,1.4125180124223602E8
Racing,2155,1.2637962877030163E8
Sports,2666,1.1815068829707427E8


Databricks visualization. Run in Databricks to view.

### 3.Platform analysis

#### 3.1 Are most games available on Windows/Mac/Linux instead?





In [0]:
# Are most games available on Windows/Mac/Linux instead?
platform_tbl = df_genres_exploded.select("name", "windows", "mac", "linux","genre")
display(platform_tbl)
# make a bar diagram counting how many games are available on wich platform
platform_tbl.groupBy("windows", "mac", "linux").count().show()
df_plat = (
    platform_tbl
    .withColumn("plateforme", F.concat_ws(" ",
        F.when(F.col("windows") == "true", "Win"),
        F.when(F.col("mac") == "true", "Mac"),
        F.when(F.col("linux") == "true", "Linux")))
    .groupBy("plateforme").count()
    .orderBy(F.desc("count"))
)

display(df_plat)



name,windows,mac,linux,genre
Counter-Strike,true,true,true,Action
ASCENXION,true,false,false,Action
ASCENXION,true,false,false,Adventure
ASCENXION,true,false,false,Indie
Crown Trick,true,false,false,Adventure
Crown Trick,true,false,false,Indie
Crown Trick,true,false,false,RPG
Crown Trick,true,false,false,Strategy
"Cook, Serve, Delicious! 3?!",true,true,false,Action
"Cook, Serve, Delicious! 3?!",true,true,false,Indie


+-------+-----+-----+------+
|windows|  mac|linux| count|
+-------+-----+-----+------+
|   true| true|false| 16535|
|   true| true| true| 19389|
|   true|false| true|  4636|
|  false| true|false|    30|
|   true|false|false|116675|
|  false|false| true|     3|
|  false| true| true|     3|
+-------+-----+-----+------+



plateforme,count
Win,116675
Win Mac Linux,19389
Win Mac,16535
Win Linux,4636
Mac,30
Mac Linux,3
Linux,3


Databricks visualization. Run in Databricks to view.

#### 3.2 Do certain genres tend to be preferentially available on certain platforms?

In [0]:
platform_genre = platform_tbl.withColumn("plateforme", F.concat_ws(" ",
        F.when(F.col("windows") == "true", "Win"),
        F.when(F.col("mac") == "true", "Mac"),
        F.when(F.col("linux") == "true", "Linux")))
display(platform_genre)


name,windows,mac,linux,genre,plateforme
Counter-Strike,true,true,true,Action,Win Mac Linux
ASCENXION,true,false,false,Action,Win
ASCENXION,true,false,false,Adventure,Win
ASCENXION,true,false,false,Indie,Win
Crown Trick,true,false,false,Adventure,Win
Crown Trick,true,false,false,Indie,Win
Crown Trick,true,false,false,RPG,Win
Crown Trick,true,false,false,Strategy,Win
"Cook, Serve, Delicious! 3?!",true,true,false,Action,Win Mac
"Cook, Serve, Delicious! 3?!",true,true,false,Indie,Win Mac


Databricks visualization. Run in Databricks to view.

In [0]:
# make a table by platform


  
  
## Helpers 🦮

To help you achieve this project, here are a few tips that should help you:

    To adopt different levels of analysis, it might be useful to create different dataframes.

    As the dataset is semi-structured with a nested schema, Pyspark's methods such as getField() and explode() may help you.

    There are some text and date fields in this dataset: Pyspark offers utilitary functions to manipulate these types of data efficiently 💡

    You can use agregate functions and groupBy to conduct segmented analysis.

## Deliverable 📬

To complete this project, you should deliver:

    One or several notebooks including data manipulation with PySpark and data visualization with Databrick's dashboarding tool.

    To make sure the jury can view all the visualizations, please use the "publish" button on Databricks notebooks to create a public url where a copy of your notebook will be available.

    While using the "publish" button, Databricks may tell you that your notebook's size exceeds the maximal size allowed. If this happens, just split your notebook in several notebooks.

    Please copy-paste the link(s) to your published notebooks into your Github repo such that the jury can access it 

    
